# Week 10 Demo: Data Cleaning with pandas

## Introduction

Chapter 6 covered how to get data into a DataFrame. `pd.read_csv()`, `json.loads()`, and `requests.get()` all produce a DataFrame from a source file or service. But loading data and having data that is ready to analyze are two different things. Real datasets arrive with missing values, inconsistent text formatting, duplicate records, and columns whose types prevent calculation. Chapter 7 covers the tools for addressing those problems before analysis begins.

This demo works through a fleet registry dataset that has the kinds of problems you encounter in practice: numeric columns with placeholder strings that blocked type inference, blank fields that became `NaN` during loading, duplicate rows, status codes that need to be expanded to readable labels, and a `ship_class` column with inconsistent casing and extra whitespace. Each part of the demo introduces tools for one category of problem.

### What This Demo Covers

- Part 1: Handling Missing Data: identifying missing values with `isnull()`, removing rows with `dropna()`, filling values with `fillna()`, and converting columns to usable numeric types with `pd.to_numeric()`
- Part 2: Data Transformation: removing duplicates with `drop_duplicates()`, recoding values with `map()`, renaming columns with `rename()`, detecting outliers with boolean indexing, and binning continuous values with `pd.cut()`
- Part 3: String Manipulation: cleaning whitespace with `str.strip()`, standardizing case with `str.lower()`, and filtering with `str.contains()`

Chapter 7 in the textbook covers additional topics not in this demo: forward and back fill strategies for `fillna()`, value substitution with `replace()`, regular expressions for pattern-based transformations, dummy variables with `pd.get_dummies()`, extension data types, and categorical data. This demo focuses on the operations you will use most often.

### Before You Begin

Download `fleet_registry_v2.csv` from the course page and place it in your `week10` folder alongside this notebook. The `pd.read_csv()` calls in this demo use just the filename, which means the CSV must be in the same folder as the notebook. If you get a `FileNotFoundError`, check that the file is in the right location before running any other cells.

## Part 1: Handling Missing Data

### Loading the dataset

Start by looking at the raw file contents, then load it with pandas and inspect what came through.

In [ ]:
import numpy as np
import pandas as pd

# Display the raw file contents before loading
with open('fleet_registry_v2.csv', 'r') as f:
    print(f.read())

**Understanding the output:**

Reading the file as plain text shows exactly what is stored on disk. Look at the `ship_class` column: some entries have extra spaces around them, and the casing varies across rows. Look at the `max_speed` column: two rows have `UNKNOWN` where a number should be. Look at `crew_size` and `launch_year`: two rows have blank fields. These are the problems the demo addresses. Seeing them in the raw file first makes it clear that these are data quality issues, not something pandas introduced during loading.

In [ ]:
# Load the file and inspect what pandas produced
fleet = pd.read_csv('fleet_registry_v2.csv')
fleet.info()

**Understanding the output:**

Three things stand out in the `.info()` report.

`crew_size` and `launch_year` both show `float64` even though those values should be whole numbers. Both columns also show fewer than 15 non-null entries. This is the same behavior you saw in Week 7: when a numeric column contains even one blank field, pandas has no integer representation for "missing," so the entire column widens to `float64` to accommodate `NaN`.

`max_speed` shows a text type (`object` or `str` depending on your pandas version) and 15 non-null entries, which looks fine until you check the values. Two rows in the file have `UNKNOWN` where a speed should be. Because pandas encountered a non-numeric string in that column, it treated the entire column as text rather than numbers.

These three columns need to be fixed before any calculations on them will work.

### Identifying missing values

`isnull()` checks every cell in the DataFrame and returns `True` where a value is missing and `False` where it is not. Calling `.sum()` on that result counts the `True` values per column.

In [ ]:
print(fleet.isnull().sum())

**Understanding the output:**

`crew_size` and `launch_year` each have one missing value. `max_speed` shows zero missing values, which confirms what `.info()` suggested: those `UNKNOWN` strings loaded as text, not as missing data. pandas does not know `UNKNOWN` means "no value" unless you tell it.

### Dropping rows with missing data

`dropna()` removes rows that contain missing values. By default it removes any row where at least one column is `NaN`.

In [ ]:
print(f"Original row count: {len(fleet)}")
print(f"After dropna(): {len(fleet.dropna())}")

**Understanding the output:**

The default behavior is aggressive. A row missing only `launch_year` gets dropped even if every other column is complete. For this dataset, that means losing a row where only one field is blank. Whether that is acceptable depends on what your analysis needs.

`dropna()` returns a new DataFrame. The original `fleet` is unchanged unless you assign the result back.

`dropna(subset=...)` limits which columns trigger the drop. A row is only removed if it has a missing value in one of the specified columns.

In [ ]:
# Only drop rows missing crew_size -- keep rows missing only launch_year
fleet_crew_required = fleet.dropna(subset=['crew_size'])
print(f"After dropna(subset=['crew_size']): {len(fleet_crew_required)}")

**Understanding the output:**

This keeps any row where `crew_size` is present, even if `launch_year` is missing. Use `subset` when some columns are more critical to your analysis than others.

### Filling missing values

Instead of removing rows with missing data, you can fill them. `fillna()` replaces `NaN` with a value you provide.

In [ ]:
# Fill missing launch_year with a constant
# This is a preview -- fleet is not modified
print(fleet['launch_year'].fillna(0))

**Understanding the output:**

The blank entry is now `0.0`. This works, but the result is misleading: a launch year of 0 does not mean anything in this dataset. Filling with a constant makes sense when there is a meaningful default (for example, filling a missing sales quantity with 0 if no sale occurred). For a numeric measurement like a year or a speed, filling with the column mean or median is usually more appropriate.

In [ ]:
# Fill missing launch_year with the column mean
year_mean = fleet['launch_year'].mean()
print(f"Mean launch year: {year_mean:.1f}")
fleet['launch_year'] = fleet['launch_year'].fillna(year_mean)
print(fleet['launch_year'])

**Understanding the output:**

The mean is calculated from the non-null values and used to fill the gap. The filled value preserves the column's overall distribution rather than introducing an arbitrary placeholder.

The mean is sensitive to outliers. If one value is much larger or smaller than the rest, it pulls the mean away from the typical value. When outliers are present, the median (the middle value when sorted) is a more stable choice. To fill with the median instead, swap `.mean()` for `.median()` in the pattern above: `fleet['launch_year'].fillna(fleet['launch_year'].median())`. You will see an example of an outlier in Part 2.

### Converting columns to numeric types

`max_speed` is still a text column because the `UNKNOWN` strings blocked type inference during loading. Replacing those strings with `NaN` and converting the column to a numeric type requires one step: `pd.to_numeric()`.

In [ ]:
print(f"max_speed type before: {fleet['max_speed'].dtype}")

# errors='coerce' replaces any value that cannot be converted with NaN
fleet['max_speed'] = pd.to_numeric(fleet['max_speed'], errors='coerce')

print(f"max_speed type after: {fleet['max_speed'].dtype}")
print(fleet[['ship_name', 'max_speed']])

**Understanding the code:**

**`pd.to_numeric()`** converts a Series to a numeric type. The `errors` parameter controls what happens when a value cannot be converted.

The default behavior (`errors='raise'`) raises an exception when it encounters a non-numeric value. That would fail here because `UNKNOWN` is not a number.

`errors='coerce'` replaces any non-convertible value with `NaN` instead of raising an error. The string `UNKNOWN` becomes `NaN`, and the column type becomes `float64`. The values that were already numeric are unchanged.

Now that the column is numeric, you can fill the missing values using the same `fillna()` approach from above.

In [ ]:
speed_mean = fleet['max_speed'].mean()
fleet['max_speed'] = fleet['max_speed'].fillna(speed_mean)
print(fleet[['ship_name', 'max_speed']])

**Understanding the output:**

The two rows that had `UNKNOWN` now have the column mean as their speed. The column is fully numeric and ready for calculation.

At this point, the `max_speed` problem is resolved: the column is now `float64` and the two `UNKNOWN` entries have been replaced with the column mean. The `crew_size` and `launch_year` columns still have one missing value each. Those are addressed in Part 2, alongside the outlier detection work that needs to happen before filling them.

## Part 2: Data Transformation

This part starts with a fresh load of the file. The missing data work in Part 1 modified `fleet` in place across several cells. Starting fresh here makes the state of the DataFrame explicit.

In [ ]:
# Fresh load for Part 2 -- convert max_speed and fill missing values
fleet = pd.read_csv('fleet_registry_v2.csv')
fleet['max_speed'] = pd.to_numeric(fleet['max_speed'], errors='coerce')
fleet['max_speed'] = fleet['max_speed'].fillna(fleet['max_speed'].mean())
fleet['crew_size'] = fleet['crew_size'].fillna(fleet['crew_size'].median())
fleet['launch_year'] = fleet['launch_year'].fillna(fleet['launch_year'].median())

print(fleet)

This applies the cleaning from Part 1 in a compact block so Part 2 begins with a DataFrame that has no missing values. The numeric columns are filled, and `max_speed` has been converted from text. The string and formatting problems in `ship_class` are intentionally left for Part 3.

### Removing duplicate rows

The dataset contains two ships that appear twice: exact copies of the same row. `drop_duplicates()` removes them.

In [ ]:
print(f"Rows before: {len(fleet)}")
fleet = fleet.drop_duplicates()
print(f"Rows after: {len(fleet)}")
print(fleet)

**Understanding the output:**

`drop_duplicates()` keeps the first occurrence of each duplicate and removes the rest. The result is a DataFrame where every row is unique.

By default it compares all columns. If you only want to detect duplicates based on specific columns, pass a `subset` list. For example, `drop_duplicates(subset=['ship_name'])` would remove any row where the ship name already appeared, even if other columns differ.

### Recoding values with map()

The `status` column contains short codes: `ACT`, `RET`, and `MNT`. These are efficient to store but not readable in a report or visualization. `map()` replaces values using a dictionary.

In [ ]:
status_labels = {
    'ACT': 'Active',
    'RET': 'Retired',
    'MNT': 'Maintenance'
}

fleet['status'] = fleet['status'].map(status_labels)
print(fleet[['ship_name', 'status']])

**Understanding the code:**

`map()` takes a dictionary and replaces each value in the Series with the corresponding dictionary value. `ACT` becomes `Active`, `RET` becomes `Retired`, `MNT` becomes `Maintenance`. Values not found in the dictionary become `NaN`, so make sure your dictionary covers every value in the column.

This is different from `fillna()`, which fills blank cells. `map()` replaces existing values with new ones.

### Renaming columns

`rename()` changes column names without modifying the data.

In [ ]:
fleet = fleet.rename(columns={
    'status': 'fleet_status',
    'max_speed': 'top_speed'
})
print(fleet.columns)

**Understanding the code:**

The `columns` parameter takes a dictionary where each key is the current column name and each value is the new name. You only need to include columns you want to rename. Columns not in the dictionary are left unchanged.

Like most pandas methods, `rename()` returns a new DataFrame. Assigning back to `fleet` updates the variable.

### Detecting and handling outliers

Before binning `crew_size`, check whether the values are reasonable.

In [ ]:
print(fleet['crew_size'].describe())

**Understanding the output:**

The maximum value is `9999.0`. Every other crew size is between 12 and 200. A crew of 9999 on a small fleet registry dataset is almost certainly a data entry error rather than a real value. Before binning, that outlier needs to be addressed.

Boolean indexing identifies it:

In [ ]:
# Find rows where crew_size is unusually large
print(fleet[fleet['crew_size'] > 1000])

Once identified, you can choose to remove the row or replace the value with `NaN`. Here we replace it with `NaN` and then fill with the median. Using the median rather than the mean matters here: the mean would be distorted by that 9999 value, while the median is resistant to it.

In [ ]:
# Replace the outlier with NaN, then fill with median
fleet.loc[fleet['crew_size'] > 1000, 'crew_size'] = np.nan

crew_median = fleet['crew_size'].median()
fleet['crew_size'] = fleet['crew_size'].fillna(crew_median)

print(fleet['crew_size'].describe())

**Understanding the code:**

`fleet.loc[condition, column] = value` is **label-based assignment**. The first argument selects rows by a boolean condition, the second argument names the column, and the assignment replaces the values that match. This is the standard pandas pattern for replacing specific values in place.

After the replacement, `crew_size` has no outlier. The describe output now shows a reasonable max value, and the median is a meaningful fill for the one missing entry.

### Binning continuous values with pd.cut()

`crew_size` is a continuous numeric column. For some analyses, it is more useful to group ships into size categories than to work with the raw numbers. `pd.cut()` creates those groups.

In [ ]:
size_bins = [0, 30, 100, 300]
size_labels = ['Small', 'Medium', 'Large']

fleet['crew_category'] = pd.cut(fleet['crew_size'], bins=size_bins, labels=size_labels)
print(fleet[['ship_name', 'crew_size', 'crew_category']])

**Understanding the code:**

**`pd.cut()`** divides a continuous column into intervals based on the bin edges you provide.

`bins` defines the edges of each interval. `[0, 30, 100, 300]` creates three intervals: 0 to 30, 30 to 100, and 100 to 300. By default, the left edge is excluded and the right edge is included, so a crew of 30 falls in the first bin (Small) and a crew of 31 falls in the second (Medium).

`labels` assigns a name to each interval. There must be one fewer label than there are bin edges because labels name the spaces between edges, not the edges themselves.

The result is a new column where each ship is assigned to a size category. Categorical grouping like this is useful when you want summaries or charts that treat values as groups rather than as individual numbers.

In [ ]:
print(fleet['crew_category'].value_counts())

**Understanding the output:**

`value_counts()` shows how many ships fall into each category. This is a quick sanity check: if all ships ended up in one category, the bin edges probably need adjustment.

## Part 3: String Manipulation

The `ship_class` column has two problems that were visible from the first `.info()` call: inconsistent casing (`Explorer`, `EXPLORER`, `explorer`) and extra whitespace around some values (`  cargo  `, ` Science `). These problems mean that `Cargo`, `CARGO`, and `  cargo  ` look like three different classes to pandas even though they represent the same thing. Grouping, filtering, or counting by `ship_class` would produce wrong results until the column is cleaned.

pandas provides a **`str` accessor** for applying string methods to an entire Series column at once. Instead of writing a loop, you call `fleet['ship_class'].str.method()` and the method is applied to every value in the column.

### Cleaning whitespace

In [ ]:
print("Before strip:")
print(fleet['ship_class'].unique())

fleet['ship_class'] = fleet['ship_class'].str.strip()

print("\nAfter strip:")
print(fleet['ship_class'].unique())

**Understanding the output:**

`str.strip()` removes leading and trailing whitespace from each value. The `  cargo  ` and ` Science ` entries become `cargo` and `Science`. Values that had no extra whitespace are unchanged.

This is the same `strip()` method you have used on individual strings throughout the course. The `str` accessor applies it to every value in the column without a loop.

### Standardizing case

After stripping whitespace, the column still has `cargo`, `CARGO`, `Cargo`, `Science`, `science`, and similar variations. `str.lower()` converts every value to lowercase, making comparisons consistent.

In [ ]:
fleet['ship_class'] = fleet['ship_class'].str.lower()

print(fleet['ship_class'].value_counts())

**Understanding the output:**

The class names are now consistent lowercase strings. `value_counts()` shows the actual count for each class instead of treating `Cargo`, `CARGO`, and `cargo` as separate categories.

Other case methods follow the same pattern: `str.upper()` converts to uppercase, and `str.title()` capitalizes the first letter of each word. Use whichever produces the format your analysis requires. The important thing is that the entire column is consistent.

### Filtering with str.contains()

`str.contains()` checks whether each value in a column contains a given substring and returns a boolean Series. Combined with boolean indexing, it filters rows by a text pattern.

In [ ]:
# Find all ships whose class contains 'fighter'
fighters = fleet[fleet['ship_class'].str.contains('fighter')]
print(fighters[['ship_name', 'ship_class', 'crew_category', 'fleet_status']])

**Understanding the code:**

`str.contains('fighter')` returns `True` for every row where `ship_class` includes the substring `fighter`. Because we standardized the column to lowercase in the previous step, `'fighter'` matches all fighter ships regardless of how the original file capitalized them.

Without the case standardization from the previous step, this filter would miss ships that were entered as `Fighter` or `FIGHTER` in the original file. This is why cleaning order matters: strip first, standardize case second, then filter or group.

## Conclusion

### What You've Learned

This demo covered three categories of data cleaning operations from Chapter 7.

Handling missing data: `isnull().sum()` identifies where missing values are and how many exist. `dropna()` removes rows containing them, with `subset` limiting which columns trigger the removal. `fillna()` replaces missing values with a constant or a computed value such as the column mean or median. `pd.to_numeric(errors='coerce')` converts a text column to numeric, replacing any non-convertible strings with `NaN` so they can then be filled.

Data transformation: `drop_duplicates()` removes exact row copies. `map()` recodes values using a dictionary, replacing stored codes with readable labels. `rename()` changes column names. Boolean indexing with `.loc` assignment identifies and replaces outlier values before they distort calculations. `pd.cut()` groups a continuous numeric column into labeled intervals.

String manipulation: the `str` accessor applies standard string methods to every value in a column. `str.strip()` removes leading and trailing whitespace. `str.lower()` standardizes casing. `str.contains()` returns a boolean Series for filtering rows by substring.

The sequence matters. In this demo, removing outliers before binning and standardizing case before filtering both produced correct results because the operations were applied in the right order. Applying them out of order would have silently produced wrong results.

### What the Textbook Covers Next

Chapter 7 covers several topics not in this demo.

`fillna()` supports method-based filling with `method='ffill'` (forward fill, which propagates the last valid value forward) and `method='bfill'` (back fill). These are useful for time-ordered data where a missing value should inherit the value before or after it.

`replace()` substitutes specific values with new ones, similar to `map()` but more flexible. It accepts dictionaries, lists, and regular expressions.

The string manipulation section of the chapter covers regular expressions for pattern-based matching and substitution. Regular expressions let you match text based on structure rather than exact characters, which is useful for cleaning phone numbers, postal codes, or any field with a predictable but variable format.

`pd.get_dummies()` converts a categorical column into a set of binary indicator columns, one per category. This is a common preparation step before certain types of analysis.

The chapter also covers extension data types (Section 7.3) and categorical data (Section 7.5), both of which address how pandas represents data internally for efficiency and correctness.